[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_03/MFM_ch3_factor_momentum/MFM_ch3_factor_momentum.ipynb)

# MFM_ch3_factor_momentum

Lecture 3, second case study (Ehsani and Linnainmaa, 2022, Factor Momentum and the Momentum Factor, Journal of Finance): mean return of seven Kenneth French factors (Mkt-RF, SMB, HML, RMW, CMA, short-term and long-term reversal) in the month after a positive or a negative prior-year return, July 1964 - July 2026; time-series factor momentum (TSFM, long the factors with a positive prior year, short the others) against stock momentum (MOM); spanning regressions with Newey-West errors.

*Modelling Financial Markets (MFM), Chapter 3: Factor Models and Asset Pricing. Author: Daniel Traian Pele.*

In [ ]:
import io
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## Data

Monthly factors from the Kenneth French Data Library: the five Fama-French factors, momentum (MOM) and the short-term (ST_Rev) and long-term (LT_Rev) reversal factors.

In [ ]:
FRENCH = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/'
FILES = {'ff5': 'F-F_Research_Data_5_Factors_2x3_CSV.zip', 'mom': 'F-F_Momentum_Factor_CSV.zip',
         'strev': 'F-F_ST_Reversal_Factor_CSV.zip', 'ltrev': 'F-F_LT_Reversal_Factor_CSV.zip'}
END = '2026-07-31'       # last month of the monthly sample used in the chapter


def french(name):
    """First table (monthly) of a Kenneth French data set, in decimals."""
    raw = urllib.request.urlopen(urllib.request.Request(FRENCH + FILES[name], headers={'User-Agent': 'Mozilla/5.0'}),
                                 timeout=120).read()
    z = zipfile.ZipFile(io.BytesIO(raw))
    lines = z.read(z.namelist()[0]).decode('latin-1').splitlines()
    i = next(k for k, l in enumerate(lines) if l.startswith(',') and len(l.split(',')) > 1)
    cols = [c.strip().replace('Mom', 'MOM') for c in lines[i].split(',')[1:]]
    rows = []
    for l in lines[i + 1:]:
        p = [x.strip() for x in l.split(',')]
        if not p or len(p[0]) != 6 or not p[0].isdigit():
            break
        rows.append([p[0]] + [float(x) for x in p[1:]])
    df = pd.DataFrame(rows, columns=['date'] + cols)
    df['date'] = pd.to_datetime(df['date'], format='%Y%m') + pd.offsets.MonthEnd(0)
    return df.set_index('date').replace([-99.99, -999.0], np.nan) / 100.0


F = pd.concat([french('ff5'), french('mom'), french('strev'), french('ltrev')], axis=1).dropna().loc[:END]
print(F.index[0].date(), F.index[-1].date(), len(F), 'months')

## Style and helpers

In [ ]:
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'savefig.facecolor': 'none',
                     'savefig.transparent': True, 'axes.grid': False, 'font.family': 'sans-serif',
                     'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'], 'font.size': 9, 'axes.labelsize': 10,
                     'axes.titlesize': 11, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.6,
                     'lines.linewidth': 1.1, 'legend.facecolor': 'none', 'legend.framealpha': 0, 'legend.fontsize': 8})
MainBlue, IDAred, Forest, Orange, Navy = '#1A3A6E', '#CD0000', '#2E7D32', '#E67E22', '#1F2A44'


def ols_hac(y, X, lags=None):
    """OLS with a constant and Newey-West (Bartlett) standard errors. Returns (coef, se, t, r2)."""
    y = np.asarray(y, float)
    X = np.column_stack([np.ones(len(y)), np.asarray(X, float).reshape(len(y), -1)])
    n = len(y)
    if lags is None:
        lags = int(np.floor(4 * (n / 100) ** (2 / 9)))
    XtXi = np.linalg.inv(X.T @ X)
    b = XtXi @ X.T @ y
    e = y - X @ b
    u = X * e[:, None]
    S = u.T @ u
    for l in range(1, lags + 1):
        G = u[l:].T @ u[:-l]
        S += (1 - l / (lags + 1)) * (G + G.T)
    se = np.sqrt(np.diag(XtXi @ S @ XtXi))
    return b, se, b / se, 1 - e.var() / y.var()

## Analysis

Signal of factor i in month t: its compounded return over months t-12 to t-1. TSFM_t = (1/7) sum_i sign(signal_i,t) r_i,t.

In [ ]:
FACS = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'ST_Rev', 'LT_Rev']   # MOM is the series to explain
FF5 = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
L = 12                                                              # lookback: months t-12 ... t-1

R = F[FACS]
S = (1 + R).rolling(L).apply(np.prod, raw=True).shift(1) - 1       # prior-year compounded return (signal)
ok = S.notna().all(axis=1)
R, S = R[ok], S[ok]
up, dn = R.where(S > 0), R.where(S <= 0)
tsfm = (np.sign(S).replace(0, -1) * R).mean(axis=1)                # time-series factor momentum
mom = F.loc[R.index, 'MOM']

# pooled difference over all factor-months, standard errors clustered by month
y = R.values.ravel() * 100
d = (S.values.ravel() > 0).astype(float)
g = np.repeat(np.arange(len(R)), len(FACS))
X = np.column_stack([np.ones_like(d), d])
XtXi = np.linalg.inv(X.T @ X)
b = XtXi @ X.T @ y
e = y - X @ b
Sg = sum(np.outer(X[g == k].T @ e[g == k], X[g == k].T @ e[g == k]) for k in range(len(R)))
se = np.sqrt(np.diag(XtXi @ Sg @ XtXi))
print(f'Sample {R.index[0]:%Y-%m} - {R.index[-1]:%Y-%m}, T = {len(R)}')
print('Mean return (% a month) after a negative / positive prior year')
print(pd.DataFrame({'after negative': dn.mean() * 100, 'after positive': up.mean() * 100}).round(2))
print(f'Average factor: {b[0]:.2f}% after a negative year, {b[0] + b[1]:.2f}% after a positive year, '
      f'difference {b[1]:.2f} pp (t = {b[1] / se[1]:.2f}, clustered by month)')


def max_dd(r):
    w = (1 + r).cumprod()
    return (w / w.cummax() - 1).min()


for name, s in [('TSFM', tsfm), ('MOM', mom)]:
    bb, _, tt, _ = ols_hac(s.values * 100, np.zeros((len(s), 0)))
    print(f'{name}: mean {bb[0]:.2f}% a month (t = {tt[0]:.2f}), annual Sharpe {s.mean() / s.std() * np.sqrt(12):.2f}, '
          f'max drawdown {max_dd(s):.0%}')
ff5 = F.loc[R.index, FF5].values * 100
b1, _, t1, _ = ols_hac(mom.values * 100, ff5)
b2, _, t2, _ = ols_hac(mom.values * 100, np.column_stack([ff5, tsfm.values * 100]))
b3, _, t3, _ = ols_hac(tsfm.values * 100, np.column_stack([ff5, mom.values * 100]))
print(f'MOM on FF5: alpha {b1[0]:.2f}% (t = {t1[0]:.2f}); on FF5 + TSFM: alpha {b2[0]:.2f}% (t = {t2[0]:.2f}), '
      f'slope on TSFM {b2[-1]:.2f} (t = {t2[-1]:.1f}); alpha {100 * (1 - b2[0] / b1[0]):.0f}% lower')
print(f'TSFM on FF5 + MOM: alpha {b3[0]:.2f}% (t = {t3[0]:.2f})')

## Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), gridspec_kw={'width_ratios': [1.25, 1]})
ax = axes[0]
names = FACS + ['Average factor']
x = np.arange(len(names))
ax.bar(x - 0.19, list(dn.mean() * 100) + [b[0]], 0.38, color=IDAred, label='After a negative prior year')
ax.bar(x + 0.19, list(up.mean() * 100) + [b[0] + b[1]], 0.38, color=Forest, label='After a positive prior year')
ax.axhline(0, color=Navy, lw=0.6)
ax.axvline(len(FACS) - 0.5, color=Navy, lw=0.5, ls=':')
ax.set_xticks(x, names, fontsize=7, rotation=30, ha='right')
ax.set_ylabel('Mean return, % a month')
ax.set_title('Factor returns after a positive or negative prior year', fontsize=9, loc='left')
ax = axes[1]
ax.plot(tsfm.index, (1 + tsfm).cumprod(), color=MainBlue, lw=1.0, label='Factor momentum (TSFM)')
ax.plot(mom.index, (1 + mom).cumprod(), color=Orange, lw=1.0, label='Stock momentum (MOM)')
ax.set_yscale('log')
ax.set_ylabel('Growth of 1 (log scale)')
ax.set_title(f'Growth of 1, {R.index[0]:%Y-%m} to {R.index[-1]:%Y-%m}', fontsize=9, loc='left')
plt.tight_layout()
fig.canvas.draw()
r = fig.canvas.get_renderer()
ymin = min(a.get_tightbbox(r).transformed(fig.transFigure.inverted()).y0 for a in fig.axes)
h1, l1 = axes[0].get_legend_handles_labels()
h2, l2 = axes[1].get_legend_handles_labels()
fig.legend(h1 + h2, l1 + l2, loc='upper center', bbox_to_anchor=(0.5, ymin - 0.01), ncol=2, frameon=False)
plt.savefig('ch3_factor_momentum.pdf', bbox_inches='tight', transparent=True)
plt.savefig('ch3_factor_momentum.png', bbox_inches='tight', transparent=True, dpi=180)
plt.show()

![ch3_factor_momentum](./ch3_factor_momentum.png)

Output: `ch3_factor_momentum.pdf`